# 00 - Smoke test
Load the VLM, render ONE BoolQ item, run it in **image** mode, print the answer.
Also reports GPU count + VRAM so you can confirm it fits on 2×T4 before Phase 2.

Run top-to-bottom on Kaggle (2×T4) or Colab (T4). No paid GPU needed.

In [ ]:
# --- setup: repo root on path + deps ---
import os, sys, subprocess
REPO = os.path.abspath('..')  # notebook lives in notebooks/
if REPO not in sys.path:
    sys.path.insert(0, REPO)

# Install everything EXCEPT torch (Kaggle/Colab ship torch+CUDA already).
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers>=4.49.0', 'accelerate>=0.34.0', 'datasets',
                'qwen-vl-utils', 'typst', 'Pillow', 'scikit-learn',
                'matplotlib', 'tqdm'], check=True)
print('deps ok')

In [ ]:
# --- GPU / VRAM report ---
import torch
print('CUDA available:', torch.cuda.is_available())
n = torch.cuda.device_count()
print('GPU count:', n)
total = 0
for i in range(n):
    p = torch.cuda.get_device_properties(i)
    gb = p.total_memory / 1e9
    total += gb
    print(f'  cuda:{i}  {p.name}  {gb:.1f} GB')
print(f'TOTAL VRAM: {total:.1f} GB')
assert torch.cuda.is_available(), 'No GPU. Switch runtime to a GPU accelerator.'

In [ ]:
# --- typst backend check (rendering.py uses the Python binding, not a CLI) ---
import typst
print('typst python binding OK:', typst.__file__)

In [ ]:
# --- render ONE item and show it (with a non-blank sanity check) ---
import config
from src.inference import load_items
from src.rendering import render_qa_to_image, image_dims, is_blank_image
import io
from PIL import Image

items = load_items(n=1)
item = items[0]
print('gold:', item['gold'])
print('question:', item['question'])
png = render_qa_to_image(item['passage'], item['question'])
print('image dims (expected):', image_dims(), '| bytes:', len(png))
assert not is_blank_image(png), (
    "Rendered PNG is blank -> Typst font fallback likely failed. "
    "Set config.RENDER['font'] to a bundled family (e.g. 'Libertinus Serif') "
    "or run: !apt-get -y install fonts-dejavu")
print('render OK: image contains ink')
Image.open(io.BytesIO(png))

In [ ]:
# --- load model (this is the VRAM moment) ---
from src.inference import load_vl_model
model, processor = load_vl_model()
for i in range(torch.cuda.device_count()):
    used = torch.cuda.memory_allocated(i) / 1e9
    print(f'cuda:{i} allocated after load: {used:.1f} GB')

In [ ]:
# --- run the single item in IMAGE mode ---
from src.inference import run_inference
res = run_inference([item], mode='image', model=model, processor=processor)[0]
print('prediction :', res['prediction'])
print('gold       :', res['gold'])
print('correct    :', bool(res['correct']))
print('raw_text   :', res['raw_text'])
print('input_toks :', res['input_tokens'], '| total_toks:', res['total_tokens'])

In [ ]:
# --- peak VRAM after a forward+generate (the number that must fit) ---
for i in range(torch.cuda.device_count()):
    peak = torch.cuda.max_memory_allocated(i) / 1e9
    print(f'cuda:{i} PEAK allocated: {peak:.1f} GB')
print('\nIf peak per GPU < ~15 GB you are safe on 2×T4. Report these numbers back.')